# 01 — Pythonic Code & the Data Model
**References:** Ramalho *Fluent Python* Ch. 1, 11–14 · Python docs: Data Model

## Narrative thread
```
Python data model -> Special methods -> Protocols -> Operator overloading -> Structural pattern matching
```

## 1. The Python Data Model

Python's data model is the system of **special (dunder) methods** that the interpreter calls to invoke operations on objects. When you write `a + b`, Python calls `type(a).__add__(a, b)`. This is the mechanism that makes user-defined types feel native.

**Key insight:** You don't implement an interface — you implement a **protocol** by defining the right dunder methods. Python uses duck typing: if it quacks, it's a duck.

| Category | Dunder methods | Triggered by |
|---|---|---|
| String repr | `__repr__`, `__str__`, `__format__` | `repr()`, `str()`, f-strings |
| Comparison | `__eq__`, `__lt__`, `__le__`, `__hash__` | `==`, `<`, `<=`, `hash()` |
| Arithmetic | `__add__`, `__mul__`, `__radd__`, `__iadd__` | `+`, `*`, `+=` |
| Container | `__len__`, `__getitem__`, `__contains__` | `len()`, `[]`, `in` |
| Callable | `__call__` | `obj()` |
| Context | `__enter__`, `__exit__` | `with` |
| Attribute | `__getattr__`, `__setattr__`, `__delattr__` | `.attr` access |
| Lifecycle | `__init__`, `__new__`, `__del__` | Object creation |
| Iteration | `__iter__`, `__next__` | `for`, `iter()`, `next()` |

In [ ]:
import math
import operator
from functools import total_ordering

# ── Full vector class implementing the Python data model ──────────────────
@total_ordering  # fills in __le__, __gt__, __ge__ from __eq__ + __lt__
class Vector:
    """
    Immutable n-dimensional vector demonstrating the Python data model.
    Implements: arithmetic, comparison, hashing, iteration, slicing.
    """
    __slots__ = ('_components',)  # no __dict__ -> memory efficient

    def __init__(self, *components):
        self._components = tuple(float(c) for c in components)

    # ── Representation ───────────────────────────────────────────────────
    def __repr__(self):
        comps = ', '.join(format(c, '.4g') for c in self._components)
        return f'Vector({comps})'

    def __format__(self, fmt_spec):
        if fmt_spec.endswith('p'):  # polar-like: magnitude∠angle (2D only)
            fmt_spec = fmt_spec[:-1]
            coords   = (abs(self), math.atan2(self._components[1], self._components[0]))
            return f'<{format(coords[0], fmt_spec)}, {format(math.degrees(coords[1]), fmt_spec)}°>'
        return f'Vector({', '.join(format(c, fmt_spec) for c in self._components)})'

    # ── Container protocol ───────────────────────────────────────────────
    def __len__(self):
        return len(self._components)

    def __getitem__(self, index):
        if isinstance(index, slice):
            return Vector(*self._components[index])
        return self._components[index]

    def __iter__(self):
        return iter(self._components)

    def __contains__(self, value):
        return value in self._components

    # ── Arithmetic ───────────────────────────────────────────────────────
    def _broadcast(self, other):
        if isinstance(other, Vector):
            if len(self) != len(other):
                raise ValueError(f'Dimension mismatch: {len(self)} vs {len(other)}')
            return self._components, other._components
        if isinstance(other, (int, float)):
            return self._components, (other,) * len(self)
        return NotImplemented, None

    def __add__(self, other):
        a, b = self._broadcast(other)
        if a is NotImplemented: return NotImplemented
        return Vector(*map(operator.add, a, b))

    __radd__ = __add__  # commutative: 3 + v works too

    def __mul__(self, scalar):
        if not isinstance(scalar, (int, float)): return NotImplemented
        return Vector(*(c * scalar for c in self._components))

    __rmul__ = __mul__

    def __matmul__(self, other):  # @ operator: dot product
        a, b = self._broadcast(other)
        if a is NotImplemented: return NotImplemented
        return sum(map(operator.mul, a, b))

    def __abs__(self):             # magnitude
        return math.sqrt(sum(c*c for c in self._components))

    def __neg__(self):
        return Vector(*(-c for c in self._components))

    # ── Comparison & hashing ─────────────────────────────────────────────
    def __eq__(self, other):
        if not isinstance(other, Vector): return NotImplemented
        return self._components == other._components

    def __lt__(self, other):  # compare by magnitude
        if not isinstance(other, Vector): return NotImplemented
        return abs(self) < abs(other)

    def __hash__(self):  # must implement if __eq__ defined
        return hash(self._components)

    # ── Callable: project onto unit vector ───────────────────────────────
    def __call__(self, other):
        """Project other onto self."""
        return (self @ other) / abs(self)**2 * self


# ── Demonstration ─────────────────────────────────────────────────────────
v1 = Vector(1, 2, 3)
v2 = Vector(4, 5, 6)

print(f'v1 = {v1}')
print(f'v2 = {v2}')
print(f'v1 + v2 = {v1 + v2}')
print(f'3 * v1  = {3 * v1}')
print(f'v1 @ v2 = {v1 @ v2}  (dot product)')
print(f'|v1|    = {abs(v1):.4f}')
print(f'v1 < v2 = {v1 < v2}  (by magnitude)')
print(f'v1[1:3] = {v1[1:3]}  (slicing returns Vector)')
print(f'2.0 in v1: {2.0 in v1}')
print(f'hash(v1): {hash(v1)}  (hashable -> usable as dict key)')
print(f'format: {v1:.2f}')

# Can use in sets and as dict keys because __hash__ and __eq__ are consistent
s = {v1, v2, Vector(1,2,3)}
print(f'\nSet deduplication: {len(s)} unique vectors (v1 and Vector(1,2,3) are equal)')

## 2. Protocols vs Abstract Base Classes

Python has two complementary approaches to defining interfaces:

**Static duck typing (Protocol, PEP 544):** Structural subtyping — if a class has the right methods, it satisfies the protocol at type-check time without inheriting.

**ABCs (abstract base classes):** Nominal subtyping — must explicitly inherit or register. Provide `isinstance()` checks and can supply default implementations via `__subclasshook__`.

In [ ]:
from typing import Protocol, runtime_checkable, SupportsFloat
from abc import ABC, abstractmethod
import numbers

# ── Protocol: structural subtyping ───────────────────────────────────────
@runtime_checkable
class Drawable(Protocol):
    def draw(self) -> str: ...
    def bounding_box(self) -> tuple[float, float, float, float]: ...

class Circle:
    def __init__(self, x, y, r): self.x, self.y, self.r = x, y, r
    def draw(self) -> str: return f'Circle at ({self.x},{self.y}) r={self.r}'
    def bounding_box(self): return self.x-self.r, self.y-self.r, self.x+self.r, self.y+self.r

class Rectangle:
    def __init__(self, x, y, w, h): self.x, self.y, self.w, self.h = x, y, w, h
    def draw(self) -> str: return f'Rect at ({self.x},{self.y}) {self.w}×{self.h}'
    def bounding_box(self): return self.x, self.y, self.x+self.w, self.y+self.h

def render(shape: Drawable) -> None:
    print(f'Drawing: {shape.draw()}')
    print(f'  bbox:  {shape.bounding_box()}')

shapes = [Circle(0, 0, 5), Rectangle(1, 2, 10, 4)]
for s in shapes:
    assert isinstance(s, Drawable), 'must be Drawable'
    render(s)

print()
# ── Structural pattern matching (Python 3.10+) ────────────────────────────
def describe_point(point):
    match point:
        case (0, 0):              return 'Origin'
        case (x, 0):              return f'On x-axis at {x}'
        case (0, y):              return f'On y-axis at {y}'
        case (x, y) if x == y:   return f'On diagonal at {x}'
        case (x, y):              return f'Point ({x}, {y})'
        case _:                   return 'Not a point'

for p in [(0,0), (3,0), (0,5), (4,4), (3,7)]:
    print(f'{p} -> {describe_point(p)}')

print()
# Match on class structure
def classify_shape(shape):
    match shape:
        case Circle(r=r) if r > 10:   return f'Large circle (r={r})'
        case Circle(r=r):              return f'Small circle (r={r})'
        case Rectangle(w=w, h=h) if w == h: return f'Square ({w}×{h})'
        case Rectangle(w=w, h=h):     return f'Rectangle ({w}×{h})'

for s in [Circle(0,0,3), Circle(0,0,15), Rectangle(0,0,5,5), Rectangle(0,0,3,7)]:
    print(classify_shape(s))

## 3. `__slots__`, `__init_subclass__`, and class variables

In [ ]:
import sys
import tracemalloc

# ── __slots__: memory layout comparison ──────────────────────────────────
class WithDict:
    def __init__(self, x, y): self.x, self.y = x, y

class WithSlots:
    __slots__ = ('x', 'y')
    def __init__(self, x, y): self.x, self.y = x, y

n = 1_000_000
tracemalloc.start()
objs_dict  = [WithDict(i, i*2) for i in range(n)]
snap1 = tracemalloc.take_snapshot()
del objs_dict

objs_slots = [WithSlots(i, i*2) for i in range(n)]
snap2 = tracemalloc.take_snapshot()
del objs_slots
tracemalloc.stop()

print('=== __slots__ memory comparison (1M objects) ===')
print(f'  sys.getsizeof(WithDict()):  {sys.getsizeof(WithDict(1,2))} bytes')
print(f'  sys.getsizeof(WithSlots()): {sys.getsizeof(WithSlots(1,2))} bytes')
print(f'  WithDict has __dict__:  {hasattr(WithDict(1,2), "__dict__")}')
print(f'  WithSlots has __dict__: {hasattr(WithSlots(1,2), "__dict__")}')
print(f'  WithSlots prevents dynamic attributes: no __dict__ -> fixed attribute set')

print()
# ── __init_subclass__: hook for class creation ────────────────────────────
class Plugin:
    _registry: dict[str, type] = {}

    def __init_subclass__(cls, /, name: str = '', **kwargs):
        super().__init_subclass__(**kwargs)
        if name:
            Plugin._registry[name] = cls
            print(f'  Registered plugin: {name!r} -> {cls.__name__}')

print('Defining plugins:')
class CSVPlugin(Plugin, name='csv'): pass
class JSONPlugin(Plugin, name='json'): pass
class ParquetPlugin(Plugin, name='parquet'): pass

print(f'\nRegistry: {Plugin._registry}')
print(f'Get plugin: {Plugin._registry["json"].__name__}')

## 4. Key takeaways

| Concept | Rule |
|---|---|
| Dunder methods | Implement the data model to make your types feel native |
| `__repr__` | Must be unambiguous; aim for `eval(repr(x)) == x` |
| `__hash__` | Must define if `__eq__` is defined; use `frozenset`-like tuple hash |
| `__slots__` | Saves 30–50% memory for many small objects; prevents `__dict__` |
| Protocol | Structural subtyping — no inheritance needed |
| `@total_ordering` | Only implement `__eq__` and one of `__lt__/le/gt/ge` |
| Pattern matching | Available 3.10+; matches structure, not just value |

**Next:** notebook 02 — closures, decorators, and the factory pattern.